In [ ]:
import numpy as np 
from pathlib import Path
import csv
import os
import glob
import keras
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers,models,regularizers,constraints
import seaborn as sns
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix,classification_report
from tensorflow.keras.callbacks import ReduceLROnPlateau,EarlyStopping
import time
from scipy.signal import fftconvolve
from sklearn.model_selection import train_test_split
import re
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus']=False
import math
import shutil
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold


In [ ]:
# ===========  Read STFT ============
t0 = time.time()
num_classes = 4

data_folder = 'Public_RF_closed_set_STFT'
drone_types = ['AVATA','M300','Mavic3','Pham4']
select_num = 190

def extract_num(filename):
    m = re.search(r'STFT_(\d+)\.npy',filename)
    return int(m.group(1)) if m else -1

# Reading data 
def load_spectrogram(
    data_folder,
    drone_types,
    num_classes = 4
    ):

    print('Reading data')
    all_data = []
    all_labels = []

    for idx,drone_type in enumerate(drone_types):
        label = np.zeros(num_classes)
        label[idx] = 1
        type_folder = os.path.join(data_folder,drone_type)
        if not os.path.exists(type_folder):
            print(f"{type_folder} not exist,skip")
            continue

        # read STFT
        file_list = [f for f in os.listdir(type_folder) if f.startswith('STFT_') and f.endswith('.npy')]
        file_list = sorted(file_list,key=extract_num)
        selected_files = file_list[:190]
        # print(selected_files.shape)
        for selected_file in selected_files:
            arr = np.load(os.path.join(type_folder,selected_file))
            all_data.append(arr)
            all_labels.append(label)
    all_data = np.asarray(all_data)
    all_labels = np.asarray(all_labels)

    X_data = all_data
    y_classify = all_labels

    print(f"Read completed Data shape:{X_data.shape},label shape:{y_classify.shape}")
    return X_data,y_classify

def post_process_data(
    X_data,
    y_classify,
    test_size = 0.2
):
    X_data = np.transpose(X_data,(0,2,3,1))
    
    np.random.seed(42)
    indices = np.random.permutation(X_data.shape[0])
    print(indices.shape)
    X_data = X_data[indices]
    y_classify = y_classify[indices]

    
    X_train_val_data,X_test_data,y_train_val_classify,y_test_classify = train_test_split(X_data,y_classify,test_size=test_size,shuffle=True,random_state=random_state,stratify=y_classify)
    print("\n === Data set partitioning ====")
    print(f"Train+Val: {X_train_val_data.shape}--{y_train_val_classify.shape}")
    print(f"Test: {X_test_data.shape}--{y_test_classify.shape}")
    return X_train_val_data,X_test_data,y_train_val_classify,y_test_classify



X_data,y_classify = load_spectrogram(data_folder,drone_types,num_classes)
test_size = 0.2
random_state=42
X_train_val_data,X_test_data,y_train_val_classify,y_test_classify = post_process_data(X_data,y_classify,test_size=test_size)
t1=time.time()
elapsed_time = t1-t0
print(f"Time: {elapsed_time:.4f} s")  

In [ ]:
# =====================  Establish Network ================ 
def resnet_block(inputs,filters,strides=1):
    x = layers.Conv2D(filters,3,strides = strides,padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters,3,strides = 1,padding='same')(x)
    x = layers.BatchNormalization()(x)

    if strides != 1 or inputs.shape[-1] != filters:
        shortcut = layers.Conv2D(filters,1,strides = strides,padding='same')(inputs)
        shortcut = layers.BatchNormalization()(shortcut)
    else:
        shortcut = inputs
    x = layers.Add()([x,shortcut])
    x = layers.ReLU()(x)
    return x

def build_RF_model(input_shape = (224,224,2),num_classes = 4):
    inputs = layers.Input(shape = input_shape)
    
    x = layers.Conv2D(64,7,strides=2,padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(3,strides = 2,padding = 'same')(x)

    # conv2_x
    x = resnet_block(x,64,strides = 1)
    x = resnet_block(x,64)
    # conv3_x
    x = resnet_block(x,128,strides = 2)
    x = resnet_block(x,128)
    # conv4_x
    x = resnet_block(x,256,strides = 2)
    x = resnet_block(x,256)
    # conv5_x
    x = resnet_block(x,512,strides = 2)
    x = resnet_block(x,512)
    
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes,activation='softmax',name = 'class_output')(x)

    model = keras.Model(inputs=inputs,outputs = outputs)
    return model

def make_fusion_dataset(X_data,y_classify,batch_size,shuffle=True):
    N = X_data.shape[0]
    inputs = X_data
    targets = {
        'class_output':y_classify
    }
    ds = tf.data.Dataset.from_tensor_slices((inputs,targets))
    if shuffle:
        ds = ds.shuffle(N)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


In [ ]:
def save_history_to_csv(history_data, filename,save_dir):
    with open(os.path.join(save_dir, filename), "w", newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(np.mat(history_data).transpose().tolist())
        

def save_result_fold(save_dir,history,predictions,y_val,main_model):
    # === Save data =============
    train_loss = history.history['loss']
    valid_loss = history.history['val_loss']
    plt.figure(figsize=(10, 6))
    plt.plot(train_loss, 'b', label='Training loss')
    plt.plot(valid_loss, 'r', label='Validation loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'Loss.png'))
    plt.close()
    
    Accuracy = history.history['accuracy']
    Val_Accuracy = history.history['val_accuracy']
    plt.figure(figsize=(10, 6))
    plt.plot(Accuracy, 'b', label='Training Accuracy')
    plt.plot(Val_Accuracy, 'r', label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'Accuracy.png'))
    plt.close()
    

    # save
    save_history_to_csv(history.history['loss'], 'train_loss.csv',save_dir)
    save_history_to_csv(history.history['val_loss'], 'val_loss.csv',save_dir)
    save_history_to_csv(history.history['accuracy'], 'Class_Accuracy.csv',save_dir)
    save_history_to_csv(history.history['val_accuracy'], 'Val_Class_Accuracy.csv',save_dir)


    y_pred_classes = np.argmax(predictions,axis=1)
    y_true_classes = np.argmax(y_val,axis=1)


    conf_matrix = confusion_matrix(y_true_classes, y_pred_classes)
    drone_types = ['AVATA','M300','Mavic3','Pham4']
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=drone_types, yticklabels=drone_types)
    plt.xlabel('Predict Label')
    plt.ylabel('Real Label')
    plt.title('Confusion Matrix')
    plt.savefig(os.path.join(save_dir, 'Confusion_Matrix.png'))
    plt.close()
    
    classification_rep = classification_report(y_true_classes, y_pred_classes, target_names=drone_types,digits=4)
    print("Classification_Report:\n",classification_rep)
    with open(os.path.join(save_dir, 'Classification_Report.txt'),"w") as f:
        f.write(classification_rep)
        
    main_model.save(save_dir + '/my_main_model',save_format = 'tf')
    filename = 'main_model_summary.txt'
    file_path = os.path.join(save_dir,filename)
    with open(file_path,'w') as f:
        main_model.summary(print_fn = lambda x: f.write(x+'\n'))

In [5]:
def get_model(num_classes=4):
    tf.keras.backend.clear_session()
    model = build_RF_model(input_shape=(224,224,2),num_classes=num_classes)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss={
            'class_output':tf.keras.losses.CategoricalCrossentropy()
        },
        metrics={
            'class_output':['accuracy']
        }
    )
    return model
def train_with_history(model, X_train_dict, X_val_dict, epochs, batch_size, verbose=1):
    result = model.fit(
        X_train_dict, validation_data=X_val_dict,
        epochs=epochs, batch_size=batch_size,
        verbose=verbose
    )
    return result

In [ ]:

nb_epoch = 1000
batch_size = 64
n_splits = 10


t0 = time.time()
loca = str(time.strftime('%Y-%m-%d-%H-%M-%S'))
base_path = f"Publicdata_RF_close_classify_Compare/Batchsize_{batch_size}_Nbepoch_{nb_epoch}/{loca}/"
os.makedirs(base_path,exist_ok = True)

y_trainval_cls = np.argmax(y_train_val_classify,axis=1)
print('y_trainval_cls former 20 items:',y_trainval_cls[:20])
print('Category distribution:',np.bincount(y_trainval_cls))

skf = StratifiedKFold(n_splits = n_splits,shuffle=True,random_state=random_state)
acc_list = []

print(f"\n=== {n_splits} fold cross-validation ====")

for fold,(train_idx,val_idx) in enumerate(skf.split(X_train_val_data,y_trainval_cls),1):
   
    print(f"\n==== Fold {fold}/{n_splits} ====")
    fold_loca = str(time.strftime('%Y-%m-%d-%H-%M-%S'))
    fold_path = f"{fold_loca}-fold{fold}"
    fold_dir = os.path.join(base_path,fold_path)
    os.makedirs(fold_dir,exist_ok = True)

    X_train,X_val = X_train_val_data[train_idx],X_train_val_data[val_idx]
    y_train,y_val = y_train_val_classify[train_idx],y_train_val_classify[val_idx]
    print(f"Train set Size:{X_train.shape[0]},Val set Size:{X_val.shape[0]}")
    
    main_model = get_model(num_classes)
    X_train_dict = make_fusion_dataset(X_train,y_train,batch_size=batch_size)
    X_val_dict = make_fusion_dataset(X_val,y_val,batch_size=batch_size,shuffle=False)

    history = train_with_history(main_model,X_train_dict=X_train_dict,X_val_dict=X_val_dict,epochs=nb_epoch,batch_size=batch_size,verbose=0)

    val_acc = main_model.evaluate(X_val_dict,verbose = 2)[1]
    acc_list.append(val_acc)

    print(f"Fold {fold} Validation Accuracy: {val_acc:.4f}")
    predictions = main_model.predict(X_val_dict)

    save_result_fold(fold_dir,history,predictions,y_val,main_model)
    
    msg = (f"Fold {fold} Acc: {val_acc:.4f}\n")
    with open(os.path.join(fold_dir, "fold_result.txt"), "w", encoding="utf-8") as f:
        f.write(msg)

      
t1=time.time()
elapsed_time = t1-t0
print(f"Time: {elapsed_time:.4f} s")

In [ ]:

print("\n==== ALL-fold Results ====")
acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)
print(f"Acc: {acc_mean:.4f} ± {acc_std:.4f}")


summary_msg = (
    "==== ALL-fold Results ====\n"
    f"Acc: {acc_mean:.4f} ± {acc_std:.4f}\n"
)
summary_path = os.path.join(base_path, "results_summary.txt")


with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_msg)

print(f"Save to: {summary_path}")


In [ ]:
# ======================= 2. Retrain the final model and evaluate on the test set =====================
print("\nRetrain the final model and evaluate on the test set")
X_train_val_dict = make_fusion_dataset(X_train_val_data,y_train_val_classify,batch_size=batch_size)
X_test_dict = make_fusion_dataset(X_test_data,y_test_classify,batch_size=batch_size,shuffle=False)

final_model = get_model(num_classes)
final_history = train_with_history(final_model, X_train_val_dict, X_train_val_dict,nb_epoch, batch_size, verbose=0)

test_acc = final_model.evaluate(X_test_dict, verbose=0)[1]
print(f"Test set accuracy: {test_acc:.4f}")
test_dir = os.path.join(base_path, "test")
os.makedirs(test_dir, exist_ok=True)
save_result_fold(test_dir, final_history, final_model.predict(X_test_dict), y_test_classify, final_model)

summary_msg_test = (
    "==== Test Results After ALL Fold ====\n"
    f"Test set Acc: {test_acc:.4f}\n"
)
summary_path = os.path.join(test_dir, "test_set_results_summary.txt")

with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_msg_test)

print(f"Save to: {summary_path}")

